# Vérifier la structure des données

In [1]:
from pathlib import Path
import json
from rich.tree import Tree
from rich import print as rprint

## Chargement données

In [2]:
def recuperer_recent(chemin_dossier)  -> list :
    '''
    Récupérer le chemin du document json le plus récent.
    Entrée : chemin du dossier visé
    Sortie : chemin du fichier horodaté le plus récent
    '''
    fichiers = []
    if chemin_dossier.exists():
        fichiers_ft = sorted(chemin_dossier.glob("*.json"))
        if fichiers_ft:
            fichiers.append(str(fichiers_ft[-1]))  # le plus récent  
    return fichiers[0]
#====================================================================================s
def charger_offres(chemin_fichier):
    '''
    Charger les offres du chemin du fichier json renseigné en paramètre
    '''
    try:
        with open(chemin_fichier, "r", encoding="utf-8") as f:
            offres = json.load(f)
        print(f"  {chemin_fichier} → {len(offres)} offres")
        return offres
    except Exception as e:
        print(f"  Erreur chargement {chemin_fichier} : {e}")


#====================================================================================
def json_vers_arbre(data, arbre, prefixe=""):
    """
    Construit récursivement un arbre Rich
    à partir d'un dictionnaire JSON.
    """
    if isinstance(data, dict):
        for cle, valeur in data.items():
            if isinstance(valeur, dict):
                # Sous-dictionnaire → nouveau nœud
                noeud = arbre.add(f"[bold cyan]{cle}[/] 📁")
                json_vers_arbre(valeur, noeud)

            elif isinstance(valeur, list):
                # Tableau → nouveau nœud avec indication du type
                noeud = arbre.add(f"[bold yellow]{cle}[/] 📋 [dim]({len(valeur)} éléments)[/]")
                if valeur and isinstance(valeur[0], dict):
                    json_vers_arbre(valeur[0], noeud)

            else:
                # Valeur simple → feuille
                type_valeur = type(valeur).__name__
                arbre.add(f"[green]{cle}[/] [dim]({type_valeur})[/]")

#====================================================================================
def afficher_structure_json(data, titre="Structure JSON"):
    """
    Affiche la structure d'un JSON sous forme d'arbre visuel.
    """
    arbre = Tree(f"[bold magenta]{titre}[/]")
    json_vers_arbre(data, arbre)
    rprint(arbre)

In [3]:
# ── Étape 1 : Récupération du nom du fichier le plus récent ──────────────────────────────
chemin_fichier_raw_ft = recuperer_recent(Path("../data/raw/francetravail"))
chemin_fichier_raw_wttj = recuperer_recent(Path("../data/raw/welcometothejungle"))

chemin_fichier_processed_ft = recuperer_recent(Path("../data/processed/francetravail"))
chemin_fichier_processed_wttj = recuperer_recent(Path("../data/processed/welcometothejungle"))

chemin_fichier_normalise = recuperer_recent(Path("../data/processed/normalise"))

# ── Étape 2 : Récupération des offres dans Python (liste de dictionnaires) ──────────────
# Offres brutes issues de l'API France Travail et du scraping du site Welcome To The Jungle
offres_brutes_ft =  charger_offres(chemin_fichier_raw_ft)
offres_brutes_wttj =  charger_offres(chemin_fichier_raw_wttj)

# Offres parsees - traitement minimal (suppression des balises html, etc... 
offres_parsees_ft =  charger_offres(chemin_fichier_processed_ft)
offres_parsees_wttj =  charger_offres(chemin_fichier_processed_wttj)

# Offres normalisées
offres_normalisees =  charger_offres(chemin_fichier_normalise)

  ../data/raw/francetravail/offres_20260417_091602.json → 1500 offres
  ../data/raw/welcometothejungle/offres_20260417_091616.json → 300 offres
  ../data/processed/francetravail/offres_20260417_091602.json → 1500 offres
  ../data/processed/welcometothejungle/offres_20260417_091616.json → 300 offres
  ../data/processed/normalise/offres_20260417_091635.json → 1800 offres


## Affichage structure

### Données brutes - France Travail

In [4]:
# Afficher 
titre = "structure données brutes - France Travail"
sample_file = offres_brutes_ft[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données brutes - France Travail
├── id (str)
├── intitule (str)
├── description (str)
├── dateCreation (str)
├── dateActualisation (str)
├── lieuTravail 📁
│   ├── libelle (str)
│   ├── latitude (float)
│   ├── longitude (float)
│   ├── codePostal (str)
│   └── commune (str)
├── romeCode (str)
├── romeLibelle (str)
├── appellationlibelle (str)
├── entreprise 📁
│   └── description (str)
├── typeContrat (str)
├── typeContratLibelle (str)
├── natureContrat (str)
├── experienceExige (str)
├── experienceLibelle (str)
├── salaire 📁
├── dureeTravailLibelle (str)
├── alternance (bool)
├── contact 📁
├── nombrePostes (int)
├── origineOffre 📁
│   ├── origine (str)
│   ├── urlOrigine (str)
│   └── partenaires 📋 (1 éléments)
│       ├── nom (str)
│       ├── url (str)
│       └── logo (str)
├── contexteTravail 📁
│   └── horaires 📋 (1 éléments)
├── entrepriseAdaptee (bool)
└── employeurHandiEngage (bool)

{
    'id': '1612153',
    'intitule': 'Développeur attaché foncier / Développeuse attachée foncière (H/F)',
    'description': "Key Account Manager terrain\nÀ propos de Rothelec\nRothelec c'est une entreprise à taille 
humaine implantée en Alsace depuis plus de 49 ans, avec un vrai rayonnement national.\nExpert de l'amélioration de 
l'habitat et fabricant de notre propre produit, nous accompagnons chaque jour des milliers de particuliers dans 
leurs projets d'économies d'énergie.\nAujourd'hui, dans le cadre de notre croissance, nous vous proposons bien plus
qu'un poste : un véritable accélérateur de carrière.\nEt vous êtesvous prêt(e) à changer de vie ?\n\xa0\nVotre 
semaine type chez Rothelec ?\nEn véritable entrepreneur de votre réussite :\n    * Vous organiserez votre semaine 
de rendez-vous clients et déplacements, transformant chaque kilomètre en opportunité de succès.\n    * Vous 
réaliserez des études techniques pour les clients particuliers et les accompagner dans leurs choix, en leur 
prodiguant des conseils personnalisés.\n    * Vous serez chargé(e) de vendre notre produit auprès des particuliers 
et de maintenir une relation professionnelle de long terme avec vos clients.\n    * Vous deviendrez un(e) 
ambassadeur(rice) de notre produit, en portant haut nos valeurs de qualité et d'éco-énergie dans le cadre du marché
dynamique de l'amélioration de l'habitat.\n    * Vous serez responsable du développement de votre secteur exclusif,
via des actions de prospection téléphonique à partir d'un fichier de prospects qualifiés mis à votre 
disposition.\n\xa0\nAucune expérience n'est exigée !\nCe que nous recherchons avant tout :\n    * Votre motivation 
et votre envie d'apprendre\xa0: nous vous transmettons l'ensemble des compétences techniques et commerciales 
indispensables.\n    * Votre aisance commerciale ou un tempérament tourné vers la relation client.\n    * Votre 
ambition : l'envie de réussir, progresser et vous dépasser.\n\xa0\nVotre rémunération & avantages\n    * 
Rémunération déplafonnée\xa0:\nNos commerciaux confirmés gagnent en moyenne 4 000 € / mois bruts,\nles meilleurs 
dépassent 8 000 € bruts.\n    * Mutuelle + participation aux bénéfices.\n    * Outils & matériel de qualité : 
véhicule, tablette\n    * Séminaires annuels + voyages du club Elite pour les meilleurs.\n    * Une entreprise 
solide, conviviale et ambitieuse.\n\xa0\nVotre parcours recrutement\n1.\xa0\xa0\xa0\xa0\xa0 Votre CV est lu 
attentivement par notre équipe recrutement.\n2.\xa0\xa0\xa0\xa0\xa0 Vous recevez un test par mail (15mn) pour 
découvrir votre profil commercial - vous obtenez aussi vos résultats par mail.\n3.\xa0\xa0\xa0\xa0\xa0 Vous êtes 
contacté(e) par la Chargée de recrutement pour un premier échange téléphonique.\n4.\xa0\xa0\xa0\xa0\xa0 Si votre 
candidature est retenue, vous passez un entretien en visioconférence.\n\xa0\nPrêt(e) à révéler votre potentiel 
?\nFaites le premier pas : postulez aujourd'hui et devenez l'un de nos futurs talents Rothelec !\n",
    'dateCreation': '2026-04-17T08:05:47.000Z',
    'dateActualisation': '2026-04-17T08:05:47.000Z',
    'lieuTravail': {
        'libelle': '67 - Strasbourg',
        'latitude': 48.56653,
        'longitude': 7.725705,
        'codePostal': '67000',
        'commune': '67482'
    },
    'romeCode': 'F1133',
    'romeLibelle': "Chargé / Chargée d'affaires foncières",
    'appellationlibelle': 'Développeur attaché foncier / Développeuse attachée foncière',
    'entreprise': {
        'description': "Rothelec est une entreprise à taille humaine implantée au cœur de l'Alsace avec un fort 
ancrage national. Depuis plus de 48 ans, nous sommes experts de l'amélioration de l'habitat et fabricant de nos 
propres produits.\nEn tant qu'acteur majeur de la transition énergétique et du confort, nous sommes reconnus pour 
notre excellence et la satisfaction de nos clients.\nSur un marché en pleine expansion, nous justifions de valeurs 
fortes que sont l'Engagement, la Confiance, le Plais

### Données brutes - Welcome to the Jungle

In [5]:
# Afficher 
titre = "structure données brutes - Welcome to the Jungle"
sample_file = offres_brutes_wttj[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données brutes - Welcome to the Jungle
├── published_at (str)
├── contract_duration_maximum (NoneType)
├── rank_group_1 (int)
├── salary_period (str)
├── _geoloc 📋 (1 éléments)
│   ├── lat (float)
│   └── lng (float)
├── reference (str)
├── profile_ranking (int)
├── offices 📋 (1 éléments)
│   ├── state (str)
│   ├── city (str)
│   ├── country (str)
│   ├── country_code (str)
│   ├── district (str)
│   ├── local_district (str)
│   ├── local_city (str)
│   └── local_state (str)
├── has_experience_level_minimum (bool)
├── published_at_date (str)
├── has_benefits (bool)
├── organization_score (int)
├── summary (str)
├── language (str)
├── sectors 📋 (3 éléments)
│   ├── name (str)
│   ├── reference (str)
│   ├── parent_name (str)
│   └── parent_reference (str)
├── is_boosted (bool)
├── remote (str)
├── has_salary_yearly_minimum (bool)
├── published_at_timestamp (int)
├── salary_maximum (int)
├── wk_reference (str)
├── rank_group_3 (int)
├── organization 📁
│   ├── name (str)
│   ├── description (NoneType)
│   ├── reference (str)
│   ├── labels 📋 (2 éléments)
│   ├── summary (str)
│   ├── profile_type (str)
│   ├── slug (str)
│   ├── logo 📁
│   │   ├── url (str)
│   │   └── thumb 📁
│   │       └── url (str)
│   ├── nb_employees (int)
│   ├── creation_year (int)
│   ├── cover_image 📁
│   │   ├── small 📁
│   │   │   └── url (str)
│   │   ├── url (str)
│   │   ├── medium 📁
│   │   │   └── url (str)
│   │   ├── large 📁
│   │   │   └── url (str)
│   │   └── social 📁
│   │       └── url (str)
│   ├── profile_ranking (int)
│   ├── commitments 📋 (3 éléments)
│   └── equality_index (NoneType)
├── salary_currency (str)
├── new_profession 📁
│   ├── sub_category_reference (str)
│   ├── sub_category_name (str)
│   ├── category_reference (str)
│   ├── category_name (str)
│   ├── pivot_name (str)
│   └── pivot_reference (str)
├── contract_duration_minimum (NoneType)
├── has_education_level (bool)
├── profile (str)
├── salary_yearly_minimum (int)
├── name (str)
├── rank_group_2 (int)
├── education_level (str)
├── has_remote (bool)
├── experience_level_minimum (int)
├── contract_type (str)
├── source_stage (str)
├── salary_minimum (int)
├── benefits 📋 (21 éléments)
├── has_contract_duration (bool)
├── key_missions 📋 (3 éléments)
├── slug (str)
├── objectID (str)
└── _highlightResult 📁
    ├── offices 📋 (1 éléments)
    │   ├── state 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── city 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── country_code 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── local_district 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   ├── local_city 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (0 éléments)
    │   └── local_state 📁
    │       ├── value (str)
    │       ├── matchLevel (str)
    │       └── matchedWords 📋 (0 éléments)
    ├── summary 📁
    │   ├── value (str)
    │   ├── matchLevel (str)
    │   ├── fullyHighlighted (bool)
    │   └── matchedWords 📋 (1 éléments)
    ├── organization 📁
    │   └── name 📁
    │       ├── value (str)
    │       ├── matchLevel (str)
    │       └── matchedWords 📋 (0 éléments)
    ├── new_profession 📁
    │   ├── sub_category_reference 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   ├── fullyHighlighted (bool)
    │   │   └── matchedWords 📋 (1 éléments)
    │   ├── sub_category_name 📁
    │   │   ├── value (str)
    │   │   ├── matchLevel (str)
    │   │   └── matchedWords 📋 (

{
    'published_at': '2026-04-15T15:03:04Z',
    'contract_duration_maximum': None,
    'rank_group_1': 6852,
    'salary_period': 'yearly',
    '_geoloc': [{'lat': 48.87646, 'lng': 2.35735}],
    'reference': '41ad49da-eb11-48dd-af63-2d292ff00c70',
    'profile_ranking': 100,
    'offices': [
        {
            'state': 'Ile-de-France',
            'city': 'Paris',
            'country': 'France',
            'country_code': 'FR',
            'district': 'Paris',
            'local_district': 'Paris',
            'local_city': 'Paris',
            'local_state': 'Île-de-France'
        }
    ],
    'has_experience_level_minimum': True,
    'published_at_date': '2026-04-15',
    'has_benefits': True,
    'organization_score': 85,
    'summary': "Rejoignez INOCO, une société de conseil tech qui valorise votre carrière et vous offre des 
opportunités d'évolution. En tant qu'Ingénieur.e Data, vous travaillerez sur des projets d'envergure dans divers 
secteurs, tout en faisant partie d'une communauté tech passionnée. Vous bénéficierez d'avantages tels qu'un titre 
de transport à 100%, une mutuelle et prévoyance à 100% pour vous et vos enfants, des primes d'intéressement, des 
jours de télétravail, et bien plus encore.",
    'language': 'fr',
    'sectors': [
        {'name': 'Logiciels', 'reference': 'software-1', 'parent_name': 'Tech', 'parent_reference': 'tech-1'},
        {
            'name': 'IT / Digital',
            'reference': 'it-digital-1',
            'parent_name': 'Conseil / Audit',
            'parent_reference': 'consulting-audit'
        },
        {'name': 'Big Data', 'reference': 'big-data-1', 'parent_name': 'Tech', 'parent_reference': 'tech-1'}
    ],
    'is_boosted': False,
    'remote': 'punctual',
    'has_salary_yearly_minimum': True,
    'published_at_timestamp': 1776265384,
    'salary_maximum': 62000,
    'wk_reference': 'INOCO_7M1YD50',
    'rank_group_3': 6852,
    'organization': {
        'name': 'INOCO',
        'description': None,
        'reference': 'NQt3T9',
        'labels': ['ecovadis-bronze', 'great-place-to-work'],
        'summary': 'Société de conseil tech indépendante pour produits et apps.',
        'profile_type': 'standard',
        'slug': 'inoco',
        'logo': {
            'url': 
'https://cdn-images.welcometothejungle.com/-dVLH7w7p_aZk28j9LOTNUklKVtTvJkXUJWN5_sLik4/rs:auto:400::/q:85/czM6Ly93d
HRqLXByb2R1Y3Rpb24vdXBsb2Fkcy9vcmdhbml6YXRpb24vbG9nby8wMDQyLzE2OTY5NS9mZjM3Yjk5MS1hNjhmLTQzODItYmEwYS1iYjdhYmI1OGU0
NGIucG5n',
            'thumb': {
                'url': 
'https://cdn-images.welcometothejungle.com/5Agcs_xqCFJZWI51U-vEQ5OLdkOoSIY5IV_KdkT0Drw/rs:auto:70::/q:85/czM6Ly93dH
RqLXByb2R1Y3Rpb24vdXBsb2Fkcy9vcmdhbml6YXRpb24vbG9nby8wMDQyLzE2OTY5NS9mZjM3Yjk5MS1hNjhmLTQzODItYmEwYS1iYjdhYmI1OGU0N
GIucG5n'
            }
        },
        'nb_employees': 65,
        'creation_year': 2018,
        'cover_image': {
            'small': {
                'url': 
'https://cdn-images.welcometothejungle.com/6DlGa6RZsyNiqAUwhkL4LRvV7ll0aMyPHF2TCQmzgAw/rs:auto:640::/q:85/czM6Ly93d
HRqLXByb2R1Y3Rpb24vdXBsb2Fkcy93ZWJzaXRlX29yZ2FuaXphdGlvbi9jb3Zlcl9pbWFnZS93dHRqX2ZyL2ZyLTc5ZjZhZjAwLWRiOGUtNGZlYS04
ZTFlLTEwYmRkY2RkOTFhYi5qcGc'
            },
            'url': 
'https://cdn-images.welcometothejungle.com/7JNztQZ6_SDTkLIw5Mn6-5hSLUFvO1EdrFyKz-doC-0/rs:auto:2000:450:/g:fp:0:0.4
4/q:85/czM6Ly93dHRqLXByb2R1Y3Rpb24vdXBsb2Fkcy93ZWJzaXRlX29yZ2FuaXphdGlvbi9jb3Zlcl9pbWFnZS93dHRqX2ZyL2ZyLTc5ZjZhZjAw
LWRiOGUtNGZlYS04ZTFlLTEwYmRkY2RkOTFhYi5qcGc',
            'medium': {
                'url': 
'https://cdn-images.welcometothejungle.com/sVfq73OIejgeIIxiZAIToebrhT6AGqmukd5ljCrP8hg/rs:auto:900::/q:85/czM6Ly93d
HRqLXByb2R1Y3Rpb24vdXBsb2Fkcy93ZWJzaXRlX29yZ2FuaXphdGlvbi9jb3Zlcl9pbWFnZS93dHRqX2ZyL2ZyLTc5ZjZhZjAwLWRiOGUtNGZlYS04
ZTFlLTEwYmRkY2RkOTFhYi5qcGc'
            },
            'large': {
                'url': 
'https://cdn-images.welcometothejungle.com/aZBflaQWrlluzkMFi92NShGmR0Xeqrg6WNHoBLP9c1s/rs:aut

### Données parsees - France Travail

In [6]:
# Afficher 
titre = "structure données parsées - France Travail"
sample_file = offres_parsees_ft[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données parsées - France Travail
├── id (str)
├── url (str)
├── titre (str)
├── description (str)
├── entreprise (NoneType)
├── nb_employes (NoneType)
├── type_contrat (str)
├── type_contrat_libelle (str)
├── alternance (bool)
├── nombre_postes (int)
├── temps_travail (str)
├── salaire_min (NoneType)
├── salaire_max (NoneType)
├── salaire_texte (NoneType)
├── experience_exige (str)
├── experience_min (NoneType)
├── qualification (str)
├── localisation_ville (str)
├── localisation_dept (str)
├── commune (str)
├── latitude (float)
├── longitude (float)
├── teletravail (NoneType)
├── secteur (str)
├── code_naf (str)
├── rome_code (str)
├── rome_libelle (str)
├── competences 📋 (0 éléments)
├── langues 📋 (0 éléments)
├── qualites 📋 (0 éléments)
├── url_postulation (NoneType)
├── date_publication (str)
├── date_actualisation (str)
├── date_extraction (str)
└── source (str)

{
    'id': '1612153',
    'url': 'https://candidat.francetravail.fr/offres/recherche/detail/1612153',
    'titre': 'Développeur attaché foncier / Développeuse attachée foncière',
    'description': "Key Account Manager terrain\nÀ propos de Rothelec\nRothelec c'est une entreprise à taille 
humaine implantée en Alsace depuis plus de 49 ans, avec un vrai rayonnement national.\nExpert de l'amélioration de 
l'habitat et fabricant de notre propre produit, nous accompagnons chaque jour des milliers de particuliers dans 
leurs projets d'économies d'énergie.\nAujourd'hui, dans le cadre de notre croissance, nous vous proposons bien plus
qu'un poste : un véritable accélérateur de carrière.\nEt vous êtesvous prêt(e) à changer de vie ?\n\xa0\nVotre 
semaine type chez Rothelec ?\nEn véritable entrepreneur de votre réussite :\n    * Vous organiserez votre semaine 
de rendez-vous clients et déplacements, transformant chaque kilomètre en opportunité de succès.\n    * Vous 
réaliserez des études techniques pour les clients particuliers et les accompagner dans leurs choix, en leur 
prodiguant des conseils personnalisés.\n    * Vous serez chargé(e) de vendre notre produit auprès des particuliers 
et de maintenir une relation professionnelle de long terme avec vos clients.\n    * Vous deviendrez un(e) 
ambassadeur(rice) de notre produit, en portant haut nos valeurs de qualité et d'éco-énergie dans le cadre du marché
dynamique de l'amélioration de l'habitat.\n    * Vous serez responsable du développement de votre secteur exclusif,
via des actions de prospection téléphonique à partir d'un fichier de prospects qualifiés mis à votre 
disposition.\n\xa0\nAucune expérience n'est exigée !\nCe que nous recherchons avant tout :\n    * Votre motivation 
et votre envie d'apprendre\xa0: nous vous transmettons l'ensemble des compétences techniques et commerciales 
indispensables.\n    * Votre aisance commerciale ou un tempérament tourné vers la relation client.\n    * Votre 
ambition : l'envie de réussir, progresser et vous dépasser.\n\xa0\nVotre rémunération & avantages\n    * 
Rémunération déplafonnée\xa0:\nNos commerciaux confirmés gagnent en moyenne 4 000 € / mois bruts,\nles meilleurs 
dépassent 8 000 € bruts.\n    * Mutuelle + participation aux bénéfices.\n    * Outils & matériel de qualité : 
véhicule, tablette\n    * Séminaires annuels + voyages du club Elite pour les meilleurs.\n    * Une entreprise 
solide, conviviale et ambitieuse.\n\xa0\nVotre parcours recrutement\n1.\xa0\xa0\xa0\xa0\xa0 Votre CV est lu 
attentivement par notre équipe recrutement.\n2.\xa0\xa0\xa0\xa0\xa0 Vous recevez un test par mail (15mn) pour 
découvrir votre profil commercial - vous obtenez aussi vos résultats par mail.\n3.\xa0\xa0\xa0\xa0\xa0 Vous êtes 
contacté(e) par la Chargée de recrutement pour un premier échange téléphonique.\n4.\xa0\xa0\xa0\xa0\xa0 Si votre 
candidature est retenue, vous passez un entretien en visioconférence.\n\xa0\nPrêt(e) à révéler votre potentiel 
?\nFaites le premier pas : postulez aujourd'hui et devenez l'un de nos futurs talents Rothelec !\n",
    'entreprise': None,
    'nb_employes': None,
    'type_contrat': 'CDI',
    'type_contrat_libelle': 'CDI',
    'alternance': False,
    'nombre_postes': 1,
    'temps_travail': '',
    'salaire_min': None,
    'salaire_max': None,
    'salaire_texte': None,
    'experience_exige': 'D',
    'experience_min': None,
    'qualification': '',
    'localisation_ville': 'Strasbourg',
    'localisation_dept': '67000',
    'commune': '67482',
    'latitude': 48.56653,
    'longitude': 7.725705,
    'teletravail': None,
    'secteur': '',
    'code_naf': '',
    'rome_code': 'F1133',
    'rome_libelle': "Chargé / Chargée d'affaires foncières",
    'competences': [],
    'langues': [],
    'qualites': [],
    'url_postulation': None,
    'date_publication': '2026-04-17',
    'date_actualisation': '2026-04-17',
    'date_extraction': '2026-04-17T09:16:16.227221',
    'source': 'francetravail'
}

### Données parsees - Welcome to the Jungle

In [7]:
# Afficher 
titre = "structure données parsées - Welcome to the Jungle"
sample_file = offres_parsees_wttj[0]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données parsées - Welcome to the Jungle
├── id (str)
├── slug (str)
├── url (str)
├── titre (str)
├── description (str)
├── missions 📋 (3 éléments)
├── type_contrat (str)
├── teletravail (str)
├── salaire_min (int)
├── salaire_max (int)
├── salaire_devise (str)
├── experience_min (int)
├── localisation_ville (str)
├── localisation_region (str)
├── pays (str)
├── latitude (float)
├── longitude (float)
├── entreprise (str)
├── entreprise_slug (str)
├── nb_employes (int)
├── entreprise_desc (str)
├── secteur (str)
├── sous_secteur (str)
├── avantages 📋 (21 éléments)
├── metier (str)
├── categorie_metier (str)
├── date_publication (str)
├── date_extraction (str)
└── source (str)

{
    'id': '41ad49da-eb11-48dd-af63-2d292ff00c70',
    'slug': 'ingenieur-e-data-data-engineer-f-h_paris',
    'url': 'https://www.welcometothejungle.com/fr/companies/inoco/jobs/ingenieur-e-data-data-engineer-f-h_paris',
    'titre': 'Ingénieur.e Data (Data Engineer) F/H',
    'description': "Rejoignez INOCO, une société de conseil tech qui valorise votre carrière et vous offre des 
opportunités d'évolution. En tant qu'Ingénieur.e Data, vous travaillerez sur des projets d'envergure dans divers 
secteurs, tout en faisant partie d'une communauté tech passionnée. Vous bénéficierez d'avantages tels qu'un titre 
de transport à 100%, une mutuelle et prévoyance à 100% pour vous et vos enfants, des primes d'intéressement, des 
jours de télétravail, et bien plus encore.\n\n* Vous êtes titulaire d’un Bac +5 (Master, MBA) ou d’un diplôme 
d’ingénieur * Vous avez une expérience significative (> 5 ans d'expérience en CDI) * Vous parlez couramment anglais
* Vous maîtrisez : Python, SQL, Spark / Databricks, Airflow / Dagster, DBT, Data Lake / Lakehouse, Kafka, ELK, 
Cloud : AWS / GCP / Azure, YAML, Ansible * Vous êtes à l’aise dans des contextes agiles (Scrum, Kanban, SAFe) * 
Vous partagez nos valeurs : * **Sens du service exceptionnel :** Vous placez les besoins des clients au cœur de vos
décisions et transformez leurs attentes en solutions concrètes et impactantes. * **Engagement envers l’excellence 
:** Vous vous démarquez par la qualité irréprochable de vos livrables et votre souci du détail, garantissant ainsi 
des résultats durables. * **Proactivité exemplaire :** Vous prenez l’initiative d’identifier les opportunités et 
proposez des solutions innovantes pour améliorer les processus et les résultats, tout en collaborant efficacement 
en équipe agile. * **Esprit d’amélioration continue :** Vous recherchez sans cesse à apprendre, à adopter les 
meilleures pratiques, et à optimiser vos méthodes de travail pour dépasser les attentes. **Pourquoi nous rejoindre 
:** * Votre carrière est prise en main de confirmé, sénior, à Lead Tech, Architecte ou Principal Engineer ! Des 
parcours d’évolution et d’implication interne vous permettront de définir votre trajectoire professionnelle selon 
vos ambitions. Avec des objectifs clairs & une augmentation salariale transparente. * Suivi mensuel de mission pour
garantir votre satisfaction. * Suivi de carrière flexible selon vos besoins : ici, votre voix compte. * Vous pouvez
travailler depuis les bureaux INOCO quand vous voulez lorsque vous êtes en mission client ! (on a développé une 
appli pour pouvoir sur synchroniser ;) ) * Vous avez 8 jours hors missions par an (en moyenne) financés par INOCO 
pour : * Participer au développement de projets interne au Agence INOCO et monter en compétences sur de nouvelles 
technos * Se former, passer des certifications (3 jours de formation en moyenne effectués chaque année) * 
Participer à l’INOCODAY, notre journée ou tous les collègues INOCO se rassemblent au siège pour animer des 
conférences tech. * Se rendre à des conférences (Devoxx, etc.) * Préparer et animer un meet up. * Participer à un 
séminaire INOCO * * * **Vous participerez :** à ce qui nous anime depuis plus de 7 ans : \\-> Accompagner 
l’innovation là où elle se trouve. \\-> Transformer le conseil dans la tech. \\-> Inscrire nos actions dans une 
démarche positive pour la société et la planète. Dès 3 ans d’entreprise, INOCO vous permet de devenir actionnaire 
associé et prendre part aux grandes décisions de l’entreprise. * * * **Nos Avantages** * Titre de transport à 100% 
* Mutuelle et prévoyance à 100% pour vous et vos enfants * Primes d'intéressement (entre 500€ et jusqu’à 2 500€ 
annuel selon les résultats). * 2-3 jours de télétravail par semaine après l'Embarquement. * 25 jours de congés + 
RTT * Chaque jour travaillé, 17 € sont ajoutés par INOCO à une cagnotte flexible (340 € net/mois) à utiliser comme 
vous voulez : loyer, électricité, cadeaux, sport, culture, CESU… * Pack télétravail 600€

### Données normalisées

In [8]:
# Afficher 
titre = "structure données normalisées"
sample_file = offres_normalisees[200]

afficher_structure_json(sample_file, titre)
rprint(sample_file)

structure données normalisées
├── id (str)
├── source (str)
├── url (str)
├── titre (str)
├── entreprise (NoneType)
├── description (str)
├── competences 📋 (0 éléments)
├── localisation_ville (str)
├── localisation_dept (str)
├── latitude (float)
├── longitude (float)
├── type_contrat (str)
├── teletravail (NoneType)
├── salaire_min (NoneType)
├── salaire_max (NoneType)
├── salaire_devise (str)
├── experience_min (int)
├── secteur (str)
├── sous_secteur (str)
├── rome_code (str)
├── nb_employes (NoneType)
├── missions 📋 (0 éléments)
├── avantages 📋 (0 éléments)
├── qualification (str)
├── date_publication (str)
└── date_extraction (str)

{
    'id': 'ft_1586218',
    'source': 'francetravail',
    'url': 'https://candidat.francetravail.fr/offres/recherche/detail/1586218',
    'titre': 'Développeur / Développeuse web mobile',
    'entreprise': None,
    'description': "Description du poste :\nMission : QA Engineer - Applications Mobiles & TV 
Connectées\nContexte\nIntégration au sein de l'équipe Applications Mobile/TV d'une plateforme vidéo de référence 
(des millions d'utilisateurs quotidiens), en mode SCRUM, au sein d'un trinôme PO/Dev/QA.\nPérimètre***Validation 
technico-fonctionnelle sur terminaux physiques (mobiles, tablettes, TV connectées : Android, iOS, AndroidTV, 
tvOS)\n* Qualification technique des bugs remontés par le support\n* Suivi des KPI qualité et reporting (crash-free
users, taux de régression)\n* Contribution à la Guilde QA, aux rituels de suivi anomalies, et à la stratégie 
d'automatisation des tests\n* Gestion du patrimoine de tests sur Xray\nDescription du profil :\nProfil 
requis***Certification ISTQB (Fondation minimum)\n* Expertise Mobile Native (iOS / Android) + gestion des Release 
Candidates\n* Maîtrise Jira / Xray\nProfil apprécié***Expertise TV Connectées (AndroidTV / tvOS)\n* Charles Proxy /
Proxyman, BrowserStack, Cucumber/Gherkin\n* Tests accessibilité RGAA (VoiceOver / TalkBack)",
    'competences': [],
    'localisation_ville': 'Paris 9E Arrondissement',
    'localisation_dept': '75009',
    'latitude': 48.872479,
    'longitude': 2.341194,
    'type_contrat': 'LIB',
    'teletravail': None,
    'salaire_min': None,
    'salaire_max': None,
    'salaire_devise': 'EUR',
    'experience_min': 6,
    'secteur': '',
    'sous_secteur': '',
    'rome_code': 'M1855',
    'nb_employes': None,
    'missions': [],
    'avantages': [],
    'qualification': '',
    'date_publication': '2026-04-16',
    'date_extraction': '2026-04-17T09:16:35.627407'
}

## Recherche clés dictionnaires

In [9]:
def recup_cles_dico(offre):
    '''
    Récupère les clés et sous-clés d'un dictionnaire de la forme [clé1, clé2__sous-clé1, clé2__sous-clé2, clé3, clé4] 
    Entrée : un dictionnaire
    Sortie : une liste de clés et sous-clés.
    '''
    cles = []
    for key, value in offre.items():   
        # Si la clé lue n'a pas été répertoriée
        if key not in cles :
            
            # Si la valeur de la clé lue est de type dictionnaire
            if isinstance(value, dict):

                # Récupération de toutes les sous-clés ET merge avec la clé actuelle pour traçabilité              
                # Mise en forme {clé : type(clé)} puis ajout à la liste des clés
                cles.extend([{f"{key}__{k}" : type(v)} for k, v in value.items()])
   
            # Sinon, Si la valeur de la clé lue est de type liste
            elif isinstance(value, list):

                #  liste ne contenant PAS de dictionnaire
                if not any(isinstance(element, dict) for element in value):
                    
                    # la clé lue est ajoutée à la liste des clés
                    cles.append({key : type(value)})

                # Si la liste contient au moins un dictionnaire (hypothèse : 1 seul dictionnaire présent dans la liste) 
                else : 
                    # Récupération de toutes les sous-clés ET merge avec la clé actuelle pour traçabilité                   
                    # Mise en forme {clé : type(clé)} puis ajout à la liste des clés
                    cles.extend([{f"{key}__{k}" : type(v)} for k, v in value[0].items()])

            # Sinon, si la clé lue n'est ni de type dictionnaire, ni de type liste
            else:
                # la clé lue est ajoutée à la liste des clés
                cles.append({key : type(value)})
    return cles

#====================================================================================s
def extraire_cles(offres):
    '''
    Récupère les clés et sous-clés d'une liste de dictionnaires de la forme [clé1, clé2__sous-clé1, clé2__sous-clé2, clé3, clé4] 
    Entrée : une liste de dictionnaires
    Sortie : une liste de clés et sous-clés.
    '''
    liste_cles = []
    for offre in offres:
        interim = recup_cles_dico(offre)
        for cle in interim:
            if cle not in liste_cles:
                liste_cles.append(cle)
    return liste_cles

In [10]:
# Clés données brutes
cles_offres_brutes_ft = extraire_cles(offres_brutes_ft)
cles_offres_brutes_wttj = extraire_cles(offres_brutes_wttj)

# Clés données parsees
cles_offres_parsees_ft = extraire_cles(offres_parsees_ft)
cles_offres_parsees_wttj = extraire_cles(offres_parsees_wttj)

# Clés données normalisées
cles_offres_normalisees = extraire_cles(offres_normalisees)

# Nombre de clés
print(f"clés données brutes - France travail - Nombre : {len(cles_offres_brutes_ft)}")
print(f"clés données brutes - Welcome to the Jungle - Nombre : {len(cles_offres_brutes_wttj)}")
print(f"clés données parsees - France travail - Nombre : {len(cles_offres_parsees_ft)}")
print(f"clés données parsees - Welcome to the Jungle - Nombre : {len(cles_offres_parsees_wttj)}")
print(f"clés données normalisées -  Nombre : {len(cles_offres_normalisees)}")

# #Affichage des clés
display(cles_offres_brutes_ft)
display(cles_offres_brutes_wttj)
display(cles_offres_parsees_ft)
display(cles_offres_parsees_wttj)
display(cles_offres_normalisees) 

clés données brutes - France travail - Nombre : 71
clés données brutes - Welcome to the Jungle - Nombre : 99
clés données parsees - France travail - Nombre : 46
clés données parsees - Welcome to the Jungle - Nombre : 38
clés données normalisées -  Nombre : 40


[{'id': str},
 {'intitule': str},
 {'description': str},
 {'dateCreation': str},
 {'dateActualisation': str},
 {'lieuTravail__libelle': str},
 {'lieuTravail__latitude': float},
 {'lieuTravail__longitude': float},
 {'lieuTravail__codePostal': str},
 {'lieuTravail__commune': str},
 {'romeCode': str},
 {'romeLibelle': str},
 {'appellationlibelle': str},
 {'entreprise__description': str},
 {'typeContrat': str},
 {'typeContratLibelle': str},
 {'natureContrat': str},
 {'experienceExige': str},
 {'experienceLibelle': str},
 {'dureeTravailLibelle': str},
 {'alternance': bool},
 {'nombrePostes': int},
 {'origineOffre__origine': str},
 {'origineOffre__urlOrigine': str},
 {'origineOffre__partenaires': list},
 {'contexteTravail__horaires': list},
 {'entrepriseAdaptee': bool},
 {'employeurHandiEngage': bool},
 {'qualificationCode': str},
 {'qualificationLibelle': str},
 {'salaire__commentaire': str},
 {'entreprise__nom': str},
 {'salaire__libelle': str},
 {'dureeTravailLibelleConverti': str},
 {'en

[{'published_at': str},
 {'contract_duration_maximum': NoneType},
 {'rank_group_1': int},
 {'salary_period': str},
 {'_geoloc__lat': float},
 {'_geoloc__lng': float},
 {'reference': str},
 {'profile_ranking': int},
 {'offices__state': str},
 {'offices__city': str},
 {'offices__country': str},
 {'offices__country_code': str},
 {'offices__district': str},
 {'offices__local_district': str},
 {'offices__local_city': str},
 {'offices__local_state': str},
 {'has_experience_level_minimum': bool},
 {'published_at_date': str},
 {'has_benefits': bool},
 {'organization_score': int},
 {'summary': str},
 {'language': str},
 {'sectors__name': str},
 {'sectors__reference': str},
 {'sectors__parent_name': str},
 {'sectors__parent_reference': str},
 {'is_boosted': bool},
 {'remote': str},
 {'has_salary_yearly_minimum': bool},
 {'published_at_timestamp': int},
 {'salary_maximum': int},
 {'wk_reference': str},
 {'rank_group_3': int},
 {'organization__name': str},
 {'organization__description': NoneType},

[{'id': str},
 {'url': str},
 {'titre': str},
 {'description': str},
 {'entreprise': NoneType},
 {'nb_employes': NoneType},
 {'type_contrat': str},
 {'type_contrat_libelle': str},
 {'alternance': bool},
 {'nombre_postes': int},
 {'temps_travail': str},
 {'salaire_min': NoneType},
 {'salaire_max': NoneType},
 {'salaire_texte': NoneType},
 {'experience_exige': str},
 {'experience_min': NoneType},
 {'qualification': str},
 {'localisation_ville': str},
 {'localisation_dept': str},
 {'commune': str},
 {'latitude': float},
 {'longitude': float},
 {'teletravail': NoneType},
 {'secteur': str},
 {'code_naf': str},
 {'rome_code': str},
 {'rome_libelle': str},
 {'competences': list},
 {'langues': list},
 {'qualites': list},
 {'url_postulation': NoneType},
 {'date_publication': str},
 {'date_actualisation': str},
 {'date_extraction': str},
 {'source': str},
 {'localisation_dept': NoneType},
 {'commune': NoneType},
 {'latitude': NoneType},
 {'longitude': NoneType},
 {'experience_min': int},
 {'entr

[{'id': str},
 {'slug': str},
 {'url': str},
 {'titre': str},
 {'description': str},
 {'missions': list},
 {'type_contrat': str},
 {'teletravail': str},
 {'salaire_min': int},
 {'salaire_max': int},
 {'salaire_devise': str},
 {'experience_min': int},
 {'localisation_ville': str},
 {'localisation_region': str},
 {'pays': str},
 {'latitude': float},
 {'longitude': float},
 {'entreprise': str},
 {'entreprise_slug': str},
 {'nb_employes': int},
 {'entreprise_desc': str},
 {'secteur': str},
 {'sous_secteur': str},
 {'avantages': list},
 {'metier': str},
 {'categorie_metier': str},
 {'date_publication': str},
 {'date_extraction': str},
 {'source': str},
 {'salaire_max': NoneType},
 {'salaire_min': NoneType},
 {'salaire_devise': NoneType},
 {'experience_min': NoneType},
 {'localisation_region': NoneType},
 {'experience_min': float},
 {'nb_employes': NoneType},
 {'secteur': NoneType},
 {'sous_secteur': NoneType}]

[{'id': str},
 {'source': str},
 {'url': str},
 {'titre': str},
 {'entreprise': NoneType},
 {'description': str},
 {'competences': list},
 {'localisation_ville': str},
 {'localisation_dept': str},
 {'latitude': float},
 {'longitude': float},
 {'type_contrat': str},
 {'teletravail': NoneType},
 {'salaire_min': NoneType},
 {'salaire_max': NoneType},
 {'salaire_devise': str},
 {'experience_min': NoneType},
 {'secteur': str},
 {'sous_secteur': str},
 {'rome_code': str},
 {'nb_employes': NoneType},
 {'missions': list},
 {'avantages': list},
 {'qualification': str},
 {'date_publication': str},
 {'date_extraction': str},
 {'localisation_dept': NoneType},
 {'latitude': NoneType},
 {'longitude': NoneType},
 {'experience_min': int},
 {'entreprise': str},
 {'salaire_min': int},
 {'salaire_max': int},
 {'nb_employes': str},
 {'teletravail': str},
 {'nb_employes': int},
 {'salaire_devise': NoneType},
 {'experience_min': float},
 {'secteur': NoneType},
 {'sous_secteur': NoneType}]